In [1]:
from GradientGang.TheFreePirate.Optimizer.OptunaOptimizer import OptunaOptimizer
from GradientGang.TheFreePirate.Optimizer.HyperParameter.Hyperparameter import *

In [2]:
dataParams = {
    'folderPath': '../../dataset/PirateProcessed',
    'trainTimeSeriesFileName': 'pirate_pain_train.csv',
    'trainGlobalFeaturesFileName': 'train_global_features.csv',
    'trainLabelsFileName': 'pirate_pain_train_labels.csv',
    'testTimeSeriesFileName': 'pirate_pain_test.csv',
    'testGlobalFeaturesFileName': 'test_global_features.csv',
    'labelsMapping': {'no_pain': 0, 'low_pain': 1, 'high_pain': 2},
    'batch_size': 64,
    'num_workers': 0,
    'columnsToIgnore': ['pain_survey_1', 'pain_survey_2', 'pain_survey_3', 'pain_survey_4', 'joint_25', 'joint_26', 'joint_00', 'joint_02', 'joint_03', 'joint_05', 'joint_06'],
}

callbacksParams = {
    'baseLogDir': 'PirateLogs',
    'earlyStoppingPatience': 20,
    'maxEpochs': 1,
}

In [6]:
numLayers = GlobalValuesHyperparameter(IntHyperparameter("globalFeaturesEncoderNumLayers", 1, 2))
embedding = GlobalValuesHyperparameter(CategoricalHyperparameter("globalFeaturesEmbeddingDim", [32, 64]))
dropout = GlobalValuesHyperparameter(FloatHyperparameter("globalFeaturesDropout", 0.0, 0.5))
useGlobal = GlobalHyperparameter( 
    hyperparameter=CategoricalHyperparameter("useGlobalFeatures", [False]),
    values=[
        numLayers,
        embedding,
        dropout
    ]
)

phi = {
    'dataParams': {
        'numFolds': 2,
        'includeTestInFolds': True,
        'augmentTestSet': True,
        'dataAugmentationParams':{
            'nCopies': 2,
            'keepOriginal': True,
            'scaleRange': 0.1,
            'jitterStdDev': 0.05,
            'offsetRange': 0.1,
            'maxWarpFraction': FloatHyperparameter("maxWartFraction", 0.0, 0.2),
            'windowSize': 10,
            'windowStride': 5,
        }
    },
    'modelParams': {
        'f1AverageStrategy': 'weighted',
        'numClasses': 3,
        'reconstructionLossWeight': 0.5,
        'useGlobalFeatures': useGlobal,
        'globalFeaturesEncoderNumLayers': numLayers,
        'globalFeaturesEmbeddingDim': embedding,
        'globalFeaturesDropout': dropout,
        'learningRate': 1e-3,
        'weightDecay': 1e-2,
        'activationFunction': 'relu',
        'timeSeriesEncoderNumLayers': 2,
        'timeSeriesEmbeddingDim': 64,
        'timeSeriesDropout': FloatHyperparameter("timeSeriesDroput", 0.0, 0.2),
        'predictorNumLayers': 2,
        'predictorDropout': 0.1,
        'aggregationStrategyForClassification': 'majorityVoting',
        'predictorFixALogit': False,
        'timeSeriesArchitectureType': 'feedForward',
    }
}

In [8]:
optim = OptunaOptimizer("NewTest_1", dataParams, callbacksParams, phi)

[I 2025-11-17 21:18:32,968] A new study created in RDB with name: NewTest_1
Seed set to 42


In [9]:
optim.optimize(1)

[PirateDataModule] Setting up K-Folds(2) with data augmentation...
[PirateDataModule] K-Folds setup completed with 2 folds.
[PirateDataModule] Setting up K-Folds(2) with data augmentation...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores


[PirateDataModule] K-Folds setup completed with 2 folds.

=== Evaluating Fold 1/2 ===


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Polimi\Master\3sem\ANN_challenges\GradientGang\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Polimi\Master\3sem\ANN_challenges\GradientGang\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


...

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=1` reached.


Validation: |          | 0/? [00:00<?, ?it/s]

Fold 1 Evaluation: 0.7302


[I 2025-11-17 21:19:18,719] Trial 0 finished with value: 0.7335297167301178 and parameters: {'maxWartFraction': 0.10976270078546496, 'useGlobalFeatures': False, 'timeSeriesDroput': 0.1430378732744839}. Best is trial 0 with value: 0.733529716730118.


In [ ]:
bestParams = optim.getBestModel()

{'dataParams': {'numFolds': 2,
  'includeTestInFolds': True,
  'augmentTestSet': True,
  'dataAugmentationParams': {'nCopies': 2,
   'keepOriginal': True,
   'scaleRange': 0.1,
   'jitterStdDev': 0.05,
   'offsetRange': 0.1,
   'maxWarpFraction': 0.109762700785465,
   'windowSize': 10,
   'windowStride': 5}},
 'modelParams': {'f1AverageStrategy': 'weighted',
  'numClasses': 3,
  'reconstructionLossWeight': 0.5,
  'useGlobalFeatures': False,
  'globalFeaturesEncoderNumLayers': None,
  'globalFeaturesEmbeddingDim': None,
  'globalFeaturesDropout': None,
  'learningRate': 0.001,
  'weightDecay': 0.01,
  'activationFunction': 'relu',
  'timeSeriesEncoderNumLayers': 2,
  'timeSeriesEmbeddingDim': 64,
  'timeSeriesDropout': 0.143037873274484,
  'predictorNumLayers': 2,
  'predictorDropout': 0.1,
  'aggregationStrategyForClassification': 'majorityVoting',
  'predictorFixALogit': False,
  'timeSeriesArchitectureType': 'feedForward'}}